In [2]:
import pandas as pd
df=pd.read_csv('train.csv')
print(df)

      Loan_ID  Gender Married Dependents     Education Self_Employed  \
0    LP001002    Male      No          0      Graduate            No   
1    LP001003    Male     Yes          1      Graduate            No   
2    LP001005    Male     Yes          0      Graduate           Yes   
3    LP001006    Male     Yes          0  Not Graduate            No   
4    LP001008    Male      No          0      Graduate            No   
..        ...     ...     ...        ...           ...           ...   
609  LP002978  Female      No          0      Graduate            No   
610  LP002979    Male     Yes         3+      Graduate            No   
611  LP002983    Male     Yes          1      Graduate            No   
612  LP002984    Male     Yes          2      Graduate            No   
613  LP002990  Female      No          0      Graduate           Yes   

     ApplicantIncome  CoapplicantIncome  LoanAmount  Loan_Amount_Term  \
0               5849                0.0         NaN           

In [ ]:
baseline_accuracy=(df['Loan_Status']=='Y').mean()
print('B-A',baseline_accuracy)
df['LoanAmount']=df['LoanAmount'].fillna(df['LoanAmount'].median())
df['Loan_Amount_Term']=df['Loan_Amount_Term'].fillna(df['Loan_Amount_Term'].median())
df['Credit_History']=df['Credit_History'].fillna(df['Credit_History'].median())
df['Gender']=df['Gender'].fillna(df['Gender'].mode()[0])
df['Married']=df['Married'].fillna(df['Married'].mode()[0])
df['Dependents']=df['Dependents'].fillna(df['Dependents'].mode()[0])
df['Self_Employed']=df['Self_Employed'].fillna(df['Self_Employed'].mode()[0])
print(df.isnull().sum())
df['Gender']=df['Gender'].map({'Male':1 ,'Female':0})
df['Married']=df['Married'].map({'Yes':1 ,'No':0})
df['Education']=df['Education'].map({'Graduate':1 ,'Not Graduate':0})
df['Self_Employed']=df['Self_Employed'].map({'Yes':1 ,'No':0})
df['Loan_Status']=df['Loan_Status'].map({'Y':1 ,'N':0})
df['Dependents']=df['Dependents'].replace('3+','3')
df['Dependents']=pd.to_numeric(df['Dependents'])
df['Property_Area'] = df['Property_Area'].map({'Urban': 0, 'Semiurban': 1, 'Rural': 2})  #PREVIOUS WORK
print(df.columns.tolist())
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
x= df[['Credit_History', 'ApplicantIncome', 'LoanAmount', 'Gender', 'Married', 'Education', 'Property_Area']]
y = df['Loan_Status']
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
model=DecisionTreeClassifier(random_state=42,max_depth=8)
model.fit(x_train,y_train)
prediction=model.predict(x_test)
model_accuracy=accuracy_score(y_test,prediction)
print('model_accuracy:',model_accuracy)
print('baseline_accuracy:',baseline_accuracy)
print('improvemnt over baqseline:',model_accuracy-baseline_accuracy)
results=x_test.copy()
results['actual']=y_test
results['predicted']=prediction

wrong_prediction=results[results['actual'] != results['predicted']]
print('wrong_prediction length',len(wrong_prediction))
print(wrong_prediction.head())
importances = pd.Series(model.feature_importances_, index=x.columns)
print(importances.sort_values(ascending=False))
print("Model accuracy:", model_accuracy)


B-A 0.6872964169381107
Loan_ID              0
Gender               0
Married              0
Dependents           0
Education            0
Self_Employed        0
ApplicantIncome      0
CoapplicantIncome    0
LoanAmount           0
Loan_Amount_Term     0
Credit_History       0
Property_Area        0
Loan_Status          0
dtype: int64
['Loan_ID', 'Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History', 'Property_Area', 'Loan_Status']
model_accuracy: 0.7398373983739838
baseline_accuracy: 0.6872964169381107
improvemnt over baqseline: 0.052540981435873046
wrong_prediction length 32
     Credit_History  ApplicantIncome  LoanAmount  Gender  Married  Education  \
340             1.0             2647       173.0       1        1          0   
77              1.0             1000       110.0       1        1          1   
209             1.0             3418       127.0       1        0          1

In [4]:
#day 22
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
baseline=y_test.value_counts(normalize=True).max()
tree=DecisionTreeClassifier(max_depth=4,random_state=42).fit(x_train,y_train)
tree_accuracy=accuracy_score(y_test,tree.predict(x_test))
print('baseline',baseline)                             
print('tree accuracy:',tree_accuracy)    

baseline 0.6504065040650406
tree accuracy: 0.7560975609756098


In [5]:
knn=KNeighborsClassifier(n_neighbors=3).fit(x_train,y_train)
unscaled_knn=accuracy_score(y_test,knn.predict(x_test))
print('unscaled_knn',unscaled_knn)

unscaled_knn 0.5853658536585366


In [6]:
scaler=StandardScaler()
x_train_s=scaler.fit_transform(x_train)
x_test_s=scaler.transform(x_test)
knn_s=KNeighborsClassifier(n_neighbors=5).fit(x_train_s,y_train)
scaled_knn=accuracy_score(y_test,knn_s.predict(x_test_s))
print('scaled knn',scaled_knn)

scaled knn 0.7886178861788617


In [7]:
for k in [1,3,5,7,9,14]:
    m=KNeighborsClassifier(n_neighbors=k).fit(x_train_s,y_train)
    print(k,round(m.score(x_train_s,y_train),3),round(m.score(x_test_s,y_test)))

1 1.0 1
3 0.859 1
5 0.837 1
7 0.817 1
9 0.809 1
14 0.811 1


In [8]:
import pandas as pd
data={
    'model':['baseline','DecisionTree','scaled_knn','unscaled_knn'],
    'Accuracy':[baseline,tree_accuracy,unscaled_knn,scaled_knn]
}
df=pd.DataFrame(data)
print(df)

          model  Accuracy
0      baseline  0.650407
1  DecisionTree  0.756098
2    scaled_knn  0.585366
3  unscaled_knn  0.788618


In [9]:
from sklearn.metrics import confusion_matrix
print('connfusion-matrix of tree',confusion_matrix(y_test,tree.predict(x_test)))
print('connfusion-matrix of knn_s',confusion_matrix(y_test,tree.predict(x_test_s)))

connfusion-matrix of tree [[16 27]
 [ 3 77]]
connfusion-matrix of knn_s [[43  0]
 [80  0]]


c:\Users\Qadri Laptop\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(
